## How to run this notebook in Google Colab

1. **Open Google Colab**: Go to [https://colab.research.google.com/](https://colab.research.google.com/)
2. **Upload this Notebook**: Click `File > Upload notebook` and select this file to open it over there, or open it directly out of GitHub/Google Drive.
3. **Run the Code**: You can run cells one by one or click `Runtime > Run all`.

# ERP-Based Stock Prediction System using LightGBM

This notebook trains a machine learning model to predict which stock items need to be refilled before they run out, based on historical inventory and sales data.

## 1. Data Loading & Exploration
First, we will load the dataset. Since this is an ERP system connected to MongoDB, we will simulate the extraction of data from the `Items` and `Sale` collections based on the database schema.

**Mapping based on your DB schema:**
- `Items`: `prodId`, `prodName`, `category`, `currentQuantity`, `unitCost`
- `Sale`: `orderId`, `date`, `products` (`productName`, `quantity`)

In [ ]:
# Colab Setup: Install required lightgbm library
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")
    !pip install lightgbm
except ImportError:
    IN_COLAB = False
    print("Not running in Colab. Assuming local environment.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

## 1.5 File Upload (Dataset Input)

If you are running this in Colab, you can provide your own target historical CSV dataset instead of using the simulated data provided in the next cell.

### **Option A: Upload manually**
Run the following cell to manually upload a local CSV file directly from your computer.

### **Option B: Load from Google Drive**
Alternatively, map your Google Drive folder and read your CSV file from there.

In [ ]:
# Option A: Upload manually (Uncomment to use)
# if IN_COLAB:
#     from google.colab import files
#     print("Upload your dataset to continue.")
#     uploaded = files.upload()
#     # Gets the filename uploaded
#     DATA_PATH = list(uploaded.keys())[0] if uploaded else "your_file.csv"
#     print(f"Dataset securely uploaded to {DATA_PATH}")

# Option B: Load from Google Drive (Uncomment to use)
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     DATA_PATH = "/content/drive/My Drive/your_dataset_folder/your_file.csv"
#     print(f"Reading from {DATA_PATH}")

In [ ]:
# Simulate extracting and merging data from MongoDB (Sale and Items collections)
np.random.seed(42)

# 1. Simulate historical daily sales data for 5 products over 365 days
dates = pd.date_range(start='2025-01-01', periods=365, freq='D')
products = ['PROD-01', 'PROD-02', 'PROD-03', 'PROD-04', 'PROD-05']

data = []
for p in products:
    # Base daily sales with some random noise and seasonality
    base_sales = np.random.poisson(lam=15, size=len(dates))
    # Simulate stock levels dropping and being restocked
    stock = 200
    for i, date in enumerate(dates):
        sales_qty = base_sales[i]
        if stock < sales_qty:
            sales_qty = stock  # Can't sell more than we have
        
        stock -= sales_qty
        
        # Record daily status
        data.append({
            'date': date,
            'prodId': p,
            'sales_quantity': sales_qty,
            'currentQuantity': stock,
            # Let's say lead time is 7 days, and safety stock is 50
            'lead_time_days': 7,
            'safety_stock': 50
        })
        
        # Restock logic (simulate buyer behavior)
        if stock <= 50:
            stock += 200 # Restock quantity

df = pd.DataFrame(data)

# Display initial data
display(df.head())
df.info()

## 2. Feature Engineering
We need to create time-based features, lag features, and rolling averages. Crucially, we must define the target variable `needs_refill`. We define a refill as needed (`1`) if the `currentQuantity` drops below a dynamic reorder point (e.g., `safety_stock + (7-day avg sales * lead_time_days)`).

In [ ]:
# Ensure data is sorted by product and date for time-series operations
df = df.sort_values(by=['prodId', 'date']).reset_index(drop=True)

# Basic time features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Lag features & Rolling averages (grouped by prodId)
df['sales_lag_1'] = df.groupby('prodId')['sales_quantity'].shift(1)
df['sales_lag_2'] = df.groupby('prodId')['sales_quantity'].shift(2)

df['sales_rolling_7d'] = df.groupby('prodId')['sales_quantity'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
df['sales_rolling_14d'] = df.groupby('prodId')['sales_quantity'].transform(lambda x: x.rolling(window=14, min_periods=1).mean())

df['stock_lag_1'] = df.groupby('prodId')['currentQuantity'].shift(1)

# Drop rows with NaN values created by shifting
df = df.dropna().reset_index(drop=True)

# Define Target Variable: needs_refill (Binary classification)
# A product needs refill if current stock is less than the expected demand over the lead time plus safety stock
expected_demand_during_lead_time = df['sales_rolling_7d'] * df['lead_time_days']
reorder_point = df['safety_stock'] + expected_demand_during_lead_time
df['needs_refill'] = (df['currentQuantity'] <= reorder_point).astype(int)

print("Target Variable Distribution:")
print(df['needs_refill'].value_counts(normalize=True))

## 3. Data Splitting
Because this is time-series data, we must not randomly shuffle the data. We use a time-based split (e.g., first 80% of days for training, last 20% for testing).

In [ ]:
features = [
    'currentQuantity', 'lead_time_days', 'safety_stock', 
    'day_of_week', 'month', 'is_weekend', 
    'sales_lag_1', 'sales_lag_2', 'sales_rolling_7d', 'sales_rolling_14d', 'stock_lag_1'
]
target = 'needs_refill'

# Time-series split parameter (using 80% for train and 20% for test)
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 4. Model Training (LightGBM)
We train a LightGBM Classifier. To handle potential class imbalances, we set `is_unbalance=True` or compute the `scale_pos_weight`.

In [ ]:
# Calculate scale_pos_weight to balance classes if necessary
neg_class = (y_train == 0).sum()
pos_class = (y_train == 1).sum()
scale_pos_weight = neg_class / pos_class if pos_class > 0 else 1.0

# Define LightGBM model parameters
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_error',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'verbosity': -1
}

model = lgb.LGBMClassifier(**lgb_params)

# Train the model
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print("Model training completed.")

## 5. Evaluation
Next, we evaluate the model using Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix on the test set.

In [ ]:
y_pred = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 6. Visualization
We visualize the Confusion Matrix and Feature Importances.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
            xticklabels=['No Refill', 'Refill Needed'],
            yticklabels=['No Refill', 'Refill Needed'])
ax1.set_title('Confusion Matrix')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')

# Feature Importance
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

sns.barplot(x='Importance', y='Feature', data=importance, ax=ax2, palette='viridis')
ax2.set_title('LightGBM Feature Importance')

plt.tight_layout()

# Save the plots before showing them
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.savefig('feature_importance.png', bbox_inches='tight')
print("Saved plots: 'confusion_matrix.png' and 'feature_importance.png'")

plt.show()

## 7. Model Saving
Save the trained model and the feature list so it can be loaded and utilized by the ERP backend for live predictions.

In [ ]:
model_data = {
    'model': model,
    'features': features
}

joblib.dump(model_data, 'stock_prediction_model.pkl')
print("Model saved successfully to 'stock_prediction_model.pkl'")

## 8. Inference Function
This function simulates how the backend would pass a single item's data to check if a refill is required.

In [ ]:
def predict_refill(item_data_dict):
    """
    Predict if an item needs an immediate refill.
    Expects a dictionary with the engineered features.
    """
    # Load model
    saved_data = joblib.load('stock_prediction_model.pkl')
    loaded_model = saved_data['model']
    required_features = saved_data['features']
    
    # Convert input to DataFrame
    input_df = pd.DataFrame([item_data_dict])
    
    # Ensure all required features are present
    for col in required_features:
        if col not in input_df.columns:
            raise ValueError(f"Missing required feature: {col}")
            
    input_features = input_df[required_features]
    
    # Predict
    prediction = loaded_model.predict(input_features)[0]
    probability = loaded_model.predict_proba(input_features)[0][1]
    
    result = {
        'needs_refill': bool(prediction),
        'probability': round(float(probability), 4)
    }
    return result

# Example Usage:
# Simulate the current state of an item (e.g., from 'Items' & 'Sale' DBs)
sample_item_state = {
    'currentQuantity': 60,
    'lead_time_days': 7,
    'safety_stock': 50,
    'day_of_week': 2,     # Wednesday
    'month': 10,          # October
    'is_weekend': 0,
    'sales_lag_1': 15,
    'sales_lag_2': 18,
    'sales_rolling_7d': 14.5,
    'sales_rolling_14d': 15.2,
    'stock_lag_1': 75
}

prediction_result = predict_refill(sample_item_state)
print(f"Inference Result: {prediction_result}")

## 9. Colab Downloads
If running inside Google Colab, executing the cell below automatically packages the generated prediction model and performance visualizations into a single `.zip` package and prompts the browser to download it.

In [ ]:
import zipfile
import os

if IN_COLAB:
    print("Running in Colab: preparing outputs for download...")
    from google.colab import files
    
    # 1. Provide instructions
    print("Zipping the saved model and plots...")

    # 2. Add our generated files into a ZIP block
    zip_name = 'stock_prediction_outputs.zip'
    files_to_zip = [
        'stock_prediction_model.pkl',
        'feature_importance.png', 
        'confusion_matrix.png'
    ]
    
    with zipfile.ZipFile(zip_name, 'w') as zf:
        for f in files_to_zip:
            if os.path.exists(f):
                zf.write(f)
            else:
                print(f"Warning: {f} not found.")
                
    # 3. Trigger download of everything in one clean gesture
    print("Download your results below!")
    files.download(zip_name)
    
else:
    print("Not running in Colab: the model and plots are already saved sequentially in your current local directory.")